In [32]:
from dotenv import load_dotenv
load_dotenv()

True

### Document Load

In [33]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader

loader = PyPDFLoader("../data/medical_report.pdf")

docs = loader.load()

print(len(docs))

9


### Document Chunk

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 20)

splitted = splitter.split_documents(docs)

print(len(splitted))

25


### Vector Embeddings

In [35]:
from langchain_community.embeddings import OpenAIEmbeddings

embed = OpenAIEmbeddings(model="text-embedding-3-small")

### Vector DB

In [36]:
from langchain_community.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore.from_documents(
    documents= splitted,
    embedding= embed 
)


### Tools

In [49]:
from langchain.tools import tool

@tool
def extract_context(query:str):
    """
    This is a retriever tool helps to extract relevant data from document.
    """
    print("tool called:", query)
    data = vector_store.similarity_search(query= query, k=4)
    context=""
    for i in data:
        context = i.page_content + '\n\n'
    return context


### LLM

In [50]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model= 'gpt-4o')


### Agent

In [51]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[extract_context],
    system_prompt= """
                    You are a helpful assistant that uses extract_context to retrieve data for external knowledge.
                    """
)

### QnA

In [52]:
query = "what is name of Patient and what is name of doctor?"
resp = agent.invoke({"messages":[{"role":"user", "content":query}]})


tool called: name of the patient
tool called: name of the doctor


In [53]:
res = resp['messages'][-1].content

print(res)

The name of the patient is Ms. Nikita Chudhary, and the name of the doctor is Dr. Nitin Nahar.
